In [1]:
import tensorflow as tf
##Select GPU 0 or 1
GPU_INDEX = 0
GPU_MEM = 0.3
###GPU Configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
gpu = '/gpu:'+ str(GPU_INDEX)
###Set Memory limit (DO NOT OVER 5G!!!)
tf.config.experimental.set_virtual_device_configuration(gpus[GPU_INDEX], 
                [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=GPU_MEM*40*1024)])
##WARNING!!!
'''You have to put some critical codes (e.g. Create ANN and fit)  
    inside the with statement(e.g. with tf.device(gpu):)!!!''';

In [2]:
def estimation(x_test,y_test,snn_mdl,F,sample_n):
    import os, sys

    class HiddenPrints:
        def __enter__(self):
            self._original_stdout = sys.stdout
            sys.stdout = open(os.devnull, 'w')

        def __exit__(self, exc_type, exc_val, exc_tb):
            sys.stdout.close()
            sys.stdout = self._original_stdout
            
    def _fun(x,y,snn_mdl,T):
        from tensorflow.keras import backend as K   
        ones = np.ones(T)
        zeros = np.zeros((sample_n-T))
        mask = np.append(ones,zeros)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=0)
        __x_test = x*mask
        _x_test_f = []
        for f_num in range(F):
            end = sample_n//F
            _x_test_f.append(tf.math.reduce_sum(__x_test[:,f_num*end:end*(f_num+1)],axis=1))
        x = tf.stack(_x_test_f,axis=1)
        _F = sample_n/F
        _F = int(np.ceil(T/_F))    
        b =0
        for f_num in range(_F):
            if f_num < _F-1:
                _t = sample_n/F
            else:
                _t = (T%(sample_n/F))
                _t = sample_n/F if _t == 0 else _t     
            timesteps = xmax*_t/(sample_n/F)
            snn_mdl.chts_model(timesteps=timesteps,
                                thresholding=0.5, 
                                scaling_factor=1,verbose=False)
            functor= K.function([snn_mdl.model.layers[0].input], [snn_mdl.model.layers[-2].output])
            b += functor(x[:,f_num])[0]
        c1 = tf.sort(b)
        c = c1[:,-1]-c1[:,-2]
        c = c.numpy()
        return b,c
    
    a = []
    b = []
    c = []
    number = 0
    for T in range(sample_n):
        with HiddenPrints():
            _b,_c = _fun(x_test,y_test,snn_mdl,T+1)
        b.append(_b)
        c.append(_c)
        print('\r Extracting Spike Gap:',number/sample_n,end='')
        print('\n')
        number += 1 

    loc = []
    for m in range(number):
        pred_right = tf.argmax(b[m],1) == tf.argmax(y_test,1)
        #pred_wrong = tf.argmax(a[m],1) != tf.argmax(y_test,1)
        for n in range(number-1-m):
            pred1_right = tf.argmax(b[m+n+1],1) == tf.argmax(y_test,1)
            #pred1_wrong = tf.argmax(a[m+n+1],1) != tf.argmax(y_test,1)
            pred_right = tf.math.logical_and(pred1_right,pred_right)
            #pred_wrong = tf.math.logical_and(pred1_wrong,pred_wrong)
        right_loc = np.where(pred_right == True)
        #wrong_loc = np.where(pred_wrong == True)
        #_loc = np.append(right_loc,wrong_loc)
        loc.append(right_loc)        
        
    hist1 = []
    hist2 = []
    for n in range(number):
        data = c[n]
        data_loc = loc[n]
        hist = tf.histogram_fixed_width(data, [0,np.max(data)], nbins=np.max(data))
        _hist = tf.histogram_fixed_width(data[data_loc], [0,np.max(data)], nbins=np.max(data))
        _hist1 = hist.numpy()  
        _hist2 = _hist.numpy()  
        hist1.append(_hist1)
        hist2.append(_hist2)
        
    l = hist1[0].shape[0]
    a_buf  = []
    a_buf2 = []
    a_buf3 = []
    a_buf4 = []
    a_buf5 = []
    a_buf6 = []
    a_buf7 = []
    
    import matplotlib.pyplot as plt
    fig,ax = plt.subplots(2,sample_n//2,figsize=(20, 10))
    x = 0
    y = 0
    confidence = []
    for n in range(number):   
        l = hist1[n].shape[0]
        _a_buf  = 1000
        _a_buf2 = 1000
        _a_buf3 = 1000
        _a_buf4 = 1000
        _a_buf5 = 1000
        _a_buf6 = 1000
        for _n in range(l):
            a = np.sum(hist1[n][l-_n-1:l]-hist2[n][l-_n-1:l])
            p = a/np.sum(hist1[n])
            p = 1 - p            
            if p == 1:
                _a_buf=l-_n-1
            if p >= 0.99:
                _a_buf2 = l-_n-1
            if p >= 0.98:
                _a_buf3 = l-_n-1
            if p >= 0.97:
                _a_buf4 = l-_n-1
            if p >= 0.96:
                _a_buf5 = l-_n-1
            if p >= 0.95:
                _a_buf6 = l-_n-1    
        a_buf.append(_a_buf)
        a_buf2.append(_a_buf2)
        a_buf3.append(_a_buf3)
        a_buf4.append(_a_buf4)
        a_buf5.append(_a_buf5)
        a_buf6.append(_a_buf6)

        
        ax[y][x].bar(np.arange(hist1[n].shape[0]),hist1[n]) 
        ax[y][x].bar(np.arange(hist2[n].shape[0]),hist2[n])
        _confidence = 1-np.sum(hist1[n]-hist2[n])/np.sum(hist1[n])
        textstr = ''.join((
        'Maximum $N_S$: %.0f\n'%(int(np.max(xmax/8*n))),
        'Largest $N_S$: %.0f\n'%(np.max(b[n])),
        'Largest $Gap$: %.0f\n'%(np.max(c[n])),
        'Confidence: %.2f\n'%(_confidence),
        '100-gap: %.2f\n'%(a_buf[-1]),
        #'995-gap: %.2f\n'%(a_buf2),
        #'99-gap: %.2f\n'%(a_buf3),
        '\n'
        )) 
        props = dict(boxstyle='round', facecolor='white', alpha=0)
        ax[y][x].text(0.4, 0.5, textstr, transform=ax[y][x].transAxes,color='black', 
                      fontsize=12, bbox=props)
        x+=1
        confidence.append(_confidence)
        if x > sample_n//2-1:
            y += 1
            x = 0
    #confidence = [np.max(a_buf),np.max(a_buf2),np.max(a_buf3),np.max(a_buf4),np.max(a_buf5),np.max(a_buf6)]
    gap = [a_buf,a_buf2,a_buf3,a_buf4,a_buf5,a_buf6]
    return confidence, gap[::-1]

def ET(mdl,_x_test,y_test,gap,F,sample_n): 
    from tensorflow.keras import backend as K
    #shape,Neuros = snn_mdl.NeuronNumbers(mode=0)
    tot_num = y_test.shape[0]
    loc1 = []
    loc1.append(np.arange(tot_num))
    continue_loc = []
    continue_loc.append(loc1)
    terminate_loc = []
    counter = []
    spiking_rate = []
    latency = []
    spike_counter = 0
    snn_mdl_ = mdl
    #te = gap
    if len(gap) == _x_test.shape[1]:
        con2 = True
    else:
        con2 = False
        
    t = 0
    for T in range(sample_n):
        T += 1 
        ones = np.ones(T)
        zeros = np.zeros((sample_n-T))
        mask = np.append(ones,zeros)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=-1)
        mask = np.expand_dims(mask,axis=0)
        __x_test = _x_test*mask
        _x_test_f = []
        for f_num in range(F):
            end = sample_n//F
            _x_test_f.append(tf.math.reduce_sum(__x_test[:,f_num*end:end*(f_num+1)],axis=1))
        x = tf.stack(_x_test_f,axis=1)             

        test = x
        label = y_test
        for _loc in continue_loc:
            test =tf.gather(test,_loc[0])
            label = tf.gather(label,_loc[0])   
        _F = sample_n/F
        _F = int(np.ceil(T/_F))    
        b =0        
        for f_num in range(_F):
            if f_num < _F-1:
                _t = sample_n/F
            else:
                _t = (T%(sample_n/F))
                _t = sample_n/F if _t == 0 else _t     
            timesteps = xmax*_t/(sample_n/F)
            snn_mdl_.chts_model(timesteps=timesteps,
                                thresholding=0.5, 
                                scaling_factor=1,verbose=False)
            functor= K.function([snn_mdl_.model.layers[0].input], [snn_mdl_.model.layers[-2].output])
            b += functor(test[:,f_num])[0] 

        #a = functor(test)[0] 
        pred_right = tf.argmax(b,1) == tf.argmax(label,1)
        pred_right = tf.cast(pred_right,"float32")
        
        c1 = tf.sort(b)
        c = c1[:,-1]-c1[:,-2]
        c = c.numpy()
        if con2:
            te = gap[t]
        else:
            te = gap
            
        try:
            con = np.max(c)<=te
        except:
            continue
            
        t += 1  
        
        if t-1 < 7 and con:
            continue
        elif t-1 < 7 and not con:
            terminate = tf.math.greater(c,te)
        elif t-1 >= 3:
            terminate = tf.math.greater(c,-1)
        
        _terminate_loc = np.where(terminate == True)
        test_spike = tf.gather(test,_terminate_loc[0])
        N = []
        for f_num in range(_F):
            if f_num < _F-1:
                _t = sample_n/F
            else:
                _t = (T%(sample_n/F))
                _t = sample_n/F if _t == 0 else _t       
            timesteps = xmax*_t/(sample_n/F)
            _,_N = snn_mdl_.SpikeCounter(test_spike[:,f_num],timesteps=timesteps,
                                        thresholding=0.5,                                        
                                        scaling_factor=1);
            N.append(_N)
        N = np.sum(_N,axis=0)
        latency.append(test_spike.shape[0]*t/sample_n)
        #for l in range(len(N)):
            #_spiking_rate.append(N[l]/(Neuros[l]*_x_test[7].shape[0]))
        right_couter = tf.gather(pred_right,_terminate_loc[0])
        right_couter = np.sum(right_couter)
        _continue_loc = np.where(terminate != True)
        terminate_loc.append(_terminate_loc[0])
        continue_loc.append(_continue_loc)
        counter.append(right_couter)
        spiking_rate.append(np.sum(N)*test_spike.shape[0]/tot_num)
        
    acc = np.sum(counter)/tot_num
    average_latency = np.sum(latency)/tot_num
    spiking_rate_ = np.mean(spiking_rate)
    early_cutoff_num = counter
    print("accuracy:",acc)      
    print("average_latency:",average_latency)      
    print("spiking_rate:",spiking_rate_)      
    return acc,spiking_rate_,average_latency,early_cutoff_num


In [ ]:
import numpy as np
from tensorflow import keras
import copy
import numpy as np
import pickle
def dvs_gesture_loader(path):
    A = list()
    B = list()

    with open(path,'rb') as pickle_file:  
        A,B = pickle.load(pickle_file)
    return A, B

path = ' '

with np.load(path) as data:
    x_train = data['x_train']
    y_train = data['y_train']
    x_test = data['x_test']
    y_test = data['y_test']
    xmax = np.max(x_train)
    y_train = keras.utils.to_categorical(y_train,11)
    y_test = keras.utils.to_categorical(y_test,11)
    
with tf.device(gpu):
    tf.random.set_seed(1000)
    train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    #train_dataset = train_dataset.map(normalisation)
    test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
    #test_dataset = test_dataset.map(normalisation)
    #AUGMENTATION

    BATCH_SIZE = 32
    SHUFFLE_BUFFER_SIZE = 1000
    #train_ds = prepare(train_ds, shuffle=True, augment=True)
    train_ds = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE,drop_remainder=True)
    val_ds = test_dataset.batch(BATCH_SIZE,drop_remainder=True)
    #T= 16
    #Data Augmentation
    #train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y),num_parallel_calls=tf.data.AUTOTUNE)
    
path1 = ' '

with np.load(path1) as data1:
    _x_train = data1['x_train']
    _xmax = np.max(_x_train)
    _y_train = data1['y_train']
    _x_test = data1['x_test']
    _y_test = data1['y_test']
    _y_test = keras.utils.to_categorical(_y_test,11)
    _y_train = keras.utils.to_categorical(_y_train,11)


In [ ]:
from spkeras import cnn_to_snn,
from spkeras.layers import Regularizer, SpikingLayer
from spkeras.layers import CFS,Clip
from tensorflow.keras import backend as K   
from spkeras.utils import save_pickle, load_pickle
from tensorflow.keras.models import load_model

with tf.device(gpu):
    for num_model in range(5):
        mdl2 = load_model('../dvs_gesture_reg_'+str(num_model)+'_4T.h5',custom_objects = {"Regularizer": Regularizer,
                                                                          "Clip":Clip,
                                                                      "SpikingLayer":SpikingLayer,
                                                                            "CFS":CFS})


        thresholding = [0.5]
        scaling_factor = [1]
        sample_n = 8
        T = list(np.arange(sample_n)+1)
        #T = [1,2,3,4,5,6,7,8]
        F = 4
        cnn_result = []
        snn_result = []

        test_dataset = tf.data.Dataset.from_tensor_slices((x_test/xmax, y_test))
        val_ds = test_dataset.batch(mdl2.layers[0].batch_size,drop_remainder=True)
        _, cnn_acc = mdl2.evaluate(val_ds)
        snn_mdl1 = cnn_to_snn(percentile=100)(mdl2,x_train/xmax)
        result = []
        for s1 in thresholding:
            _result = []
            for s2 in scaling_factor:
                __result = []
                for _T in T:
                    ones = np.ones(_T)
                    zeros = np.zeros((sample_n-_T))
                    mask = np.append(ones,zeros)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=0)
                    __x_test = _x_test*mask
                    _x_test_f = []
                    for f_num in range(F):
                        end = sample_n//F
                        _x_test_f.append(tf.math.reduce_sum(__x_test[:,f_num*end:end*(f_num+1)],axis=1))
                    __x_test = tf.stack(_x_test_f,axis=1)
                    timesteps = xmax
                    b = 0
                    _F = sample_n/F
                    _F = int(np.ceil(_T/_F))     
                    for f_num in range(_F):
                        if f_num < _F-1:
                            _t = sample_n/F
                        else:
                            _t = (_T%(sample_n/F))
                            _t = sample_n/F if _t == 0 else _t      
                        timesteps = xmax*_t/(sample_n/F)
                        snn_mdl1.chts_model(timesteps=timesteps,
                                            thresholding=s1, 
                                            scaling_factor=s2,verbose=False)
                        functor= K.function([snn_mdl1.model.layers[0].input], [snn_mdl1.model.layers[-2].output])
                        b += functor(__x_test[:,f_num])[0] 
                    pred_right = tf.argmax(b,1) == tf.argmax(_y_test,1)
                    pred_right = tf.cast(pred_right,"float32")
                    acc = np.sum(pred_right)/_y_test.shape[0]
                    tf.print(acc)                    
                    __result.append(acc)
                _result.append(__result)

            result.append(_result)

        cnn_result.append(cnn_acc)
        snn_result.append(result)   

        save_pickle(snn_result,'reg_'+str(num_model)+'_snn_result','./result/')
        save_pickle(cnn_result,'reg_'+str(num_model)+'_cnn_result','./result/') 
        c = estimation(_x_train,_y_train,snn_mdl1,F,sample_n)   
        save_pickle(c,'reg_'+str(num_model)+'_confidence','./result/')

In [ ]:
from spkeras import cnn_to_snn,
from spkeras.layers import Regularizer, SpikingLayer
from spkeras.layers import CFS,Clip
from tensorflow.keras import backend as K   
from spkeras.utils import save_pickle, load_pickle
from tensorflow.keras.models import load_model

with tf.device(gpu):
    for num_model in range(1):
        mdl2 = load_model('../dvs_gesture_clip_'+str(num_model)+'_4T.h5',custom_objects = {"Regularizer": Regularizer,
                                                                          "Clip":Clip,
                                                                      "SpikingLayer":SpikingLayer,
                                                                            "CFS":CFS})

        thresholding = [0.5]
        scaling_factor = [1]
        sample_n = 8
        T = list(np.arange(sample_n)+1)
        #T = [1,2,3,4,5,6,7,8]
        F = 4
        cnn_result = []
        snn_result = []

        test_dataset = tf.data.Dataset.from_tensor_slices((x_test/xmax, y_test))
        val_ds = test_dataset.batch(mdl2.layers[0].batch_size,drop_remainder=True)
        _, cnn_acc = mdl2.evaluate(val_ds)
        snn_mdl1 = cnn_to_snn(percentile=100)(mdl2,x_train/xmax)
        result = []
        for s1 in thresholding:
            _result = []
            for s2 in scaling_factor:
                __result = []
                for _T in T:
                    ones = np.ones(_T)
                    zeros = np.zeros((sample_n-_T))
                    mask = np.append(ones,zeros)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=-1)
                    mask = np.expand_dims(mask,axis=0)
                    __x_test = _x_test*mask
                    _x_test_f = []
                    for f_num in range(F):
                        end = sample_n//F
                        _x_test_f.append(tf.math.reduce_sum(__x_test[:,f_num*end:end*(f_num+1)],axis=1))
                    __x_test = tf.stack(_x_test_f,axis=1)
                    timesteps = xmax
                    b = 0
                    _F = sample_n/F
                    _F = int(np.ceil(_T/_F))     
                    for f_num in range(_F):
                        if f_num < _F-1:
                            _t = sample_n/F
                        else:
                            _t = (_T%(sample_n/F))
                            _t = sample_n/F if _t == 0 else _t      
                        timesteps = xmax*_t/(sample_n/F)
                        snn_mdl1.chts_model(timesteps=timesteps,
                                            thresholding=s1, 
                                            scaling_factor=s2,verbose=False)
                        functor= K.function([snn_mdl1.model.layers[0].input], [snn_mdl1.model.layers[-2].output])
                        b += functor(__x_test[:,f_num])[0] 
                    pred_right = tf.argmax(b,1) == tf.argmax(_y_test,1)
                    pred_right = tf.cast(pred_right,"float32")
                    acc = np.sum(pred_right)/_y_test.shape[0]
                    tf.print(acc)                    
                    __result.append(acc)
                _result.append(__result)

            result.append(_result)

        cnn_result.append(cnn_acc)
        snn_result.append(result)   

        save_pickle(snn_result,'clip_'+str(num_model)+'_snn_result','./result/')
        save_pickle(cnn_result,'clip_'+str(num_model)+'_cnn_result','./result/') 
        c = estimation(_x_train,_y_train,snn_mdl1,F,sample_n)   
        save_pickle(c,'clip_'+str(num_model)+'_confidence','./result/')

In [ ]:
from spkeras import cnn_to_snn,
from spkeras.layers import Regularizer, SpikingLayer
from spkeras.layers import CFS,Clip
from tensorflow.keras import backend as K   
from spkeras.utils import save_pickle, load_pickle
from tensorflow.keras.models import load_model

with tf.device(gpu):
    mdl1 = load_model('../gesture_cfs_1.h5',custom_objects = {"Regularizer": Regularizer,
                                                                             "Clip":Clip,
                                                                      "SpikingLayer":SpikingLayer,
                                                                             "CFS":CFS})
    mdl2 = load_model('../gesture_clip_1.h5',custom_objects = {"Regularizer": Regularizer,
                                                                      "SpikingLayer":SpikingLayer,
                                                                             "Clip":Clip,
                                                                             "CFS":CFS})
    mdl = load_model('../gesture_reg_1.h5',custom_objects = {"Regularizer": Regularizer,
                                                                      "SpikingLayer":SpikingLayer,
                                                                             "Clip":Clip,
                                                                             "CFS":CFS})

In [ ]:
from spkeras import cnn_to_snn

with tf.device(gpu):
    snn_mdl1 = cnn_to_snn()(mdl1,x_train)
    snn_mdl2 = cnn_to_snn()(mdl2,x_train)    
    snn_mdl = cnn_to_snn()(mdl,x_train)    

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
snn_mdl2_confidence = load_pickle('./result/clip_1_confidence.pkl')
snn_mdl1_confidence = load_pickle('./result/cfs_1_confidence.pkl')
snn_mdl_confidence = load_pickle('./result/reg_1_confidence.pkl')


c2 = np.array(snn_mdl2_confidence[1])
c1 = np.array(snn_mdl1_confidence[1])
c =  np.array(snn_mdl_confidence [1])

In [ ]:
with tf.device(gpu):
    __acc1 = []
    __spiking_rate1 = []
    __average_latency1 = []
    for gap in c1:
        _acc1,_spiking_rate1,_average_latency1 = ET(snn_mdl1,_x_test,y_test,gap,F,sample_n)
        __acc1.append(_acc1)
        __spiking_rate1.append(_spiking_rate1)                                                 
        __average_latency1.append(_average_latency1)                                                 
        save_pickle([__acc1,__spiking_rate1,__average_latency1],'cfs'+'_acc_spike_confidence','./result/')


In [ ]:
with tf.device(gpu):
    __acc2 = []
    __spiking_rate2 = []
    __average_latency2 = []
    for gap in c2:
        _acc2,_spiking_rate2,_average_latency2 = ET(snn_mdl2,_x_test,y_test,gap,F,sample_n)
        __acc2.append(_acc2)
        __spiking_rate2.append(_spiking_rate2)                                                 
        __average_latency2.append(_average_latency2)   
        save_pickle([__acc2,__spiking_rate2,__average_latency2],'clip'+'_acc_spike_confidence','./result/')

In [ ]:
with tf.device(gpu):
    __acc = []
    __spiking_rate = []
    __average_latency = []
    for gap in c:
        _acc,_spiking_rate,_average_latency = ET(snn_mdl,_x_test,y_test,gap,F,sample_n)
        __acc.append(_acc)
        __spiking_rate.append(_spiking_rate)                                                 
        __average_latency.append(_average_latency)       
        save_pickle([__acc,__spiking_rate,__average_latency],'reg'+'_acc_spike_confidence','./result/')